# Dynamics of Implied Volatility Surfaces
## Cont & da Fonseca (2002) — Implémentation complète sur le SX5E

---

> **Référence :** Rama Cont & Jose da Fonseca, *Dynamics of Implied Volatility Surfaces*, Quantitative Finance, Vol. 2, pp. 45–60, 2002.  
> **Objectif :** Décomposer les fluctuations quotidiennes de la surface de volatilité implicite en **facteurs orthogonaux** via la décomposition de Karhunen-Loève (KL), identifier leur forme, leur dynamique et leur corrélation avec le sous-jacent.

---

## Problème central résolu par ce papier

La surface de volatilité implicite $\sigma^{BS}_t(K, T)$ **évolue dynamiquement** dans le temps. Les règles déterministes utilisées par les praticiens (sticky moneyness, sticky strike) ne capturent pas ces fluctuations aléatoires.

**Questions posées :**
- Quelle est la nature de ces fluctuations ?
- Combien de facteurs suffisent pour les décrire ?
- Comment ces facteurs sont-ils corrélés avec le sous-jacent ?
- Comment modéliser et hedger le risque Vega résultant ?

**Réponse de Cont & da Fonseca :**
Un modèle à **2–3 facteurs** suffit pour expliquer >95% de la variance quotidienne, chaque facteur ayant une interprétation claire : niveau, pente (skew), convexité (butterfly).

---

## Plan du notebook

| Section | Contenu |
|---|---|
| **0** | Setup & données de marché (MDX) |
| **1** | Surface de volatilité implicite : définitions & paramétrisation |
| **2** | Lissage non-paramétrique : estimateur de Nadaraya-Watson |
| **3** | Analyse de variance : profil moyen et écart-type |
| **4** | Décomposition de Karhunen-Loève (KL) — théorie complète |
| **5** | Implémentation numérique : problème aux valeurs propres généralisé |
| **6** | Modes propres : interprétation niveau, skew, butterfly |
| **7** | Dynamique des processus de composantes principales |
| **8** | Modèle à facteurs : Ornstein-Uhlenbeck & calibration AR(1) |
| **9** | Corrélation avec le sous-jacent (effet levier) |
| **10** | Modèle factoriel complet & simulation de scénarios |
| **11** | Applications : hedging Vega, risque volatilité |
| **12** | Dashboard & synthèse |


---
## Section 0 — Setup & données

In [ ]:
# ============================================================
#  IMPORTS
# ============================================================
import warnings, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from scipy.stats import norm, kurtosis as scipy_kurtosis, skew as scipy_skew
from scipy.linalg import eigh
from scipy.optimize import brentq
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.ar_model import AutoReg
from pandas.tseries.offsets import BDay

warnings.simplefilter('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

cache_dir = Path('./cache')
cache_dir.mkdir(exist_ok=True)
print('Imports OK')

In [ ]:
# ============================================================
#  CONNEXION MDX
# ============================================================
import ezmdx
from maxxpy.apis.mdx.api import MdxClient

LOGIN_MDX    = ""   # <-- TON LOGIN
PASSWORD_MDX = ""   # <-- TON MOT DE PASSE

MDX_TYPES = {
    'volatility': 'EQUITY_VOLATILITY',
    'spot': ['STOCK_QUOTE', 'INDEX_QUOTE', 'FUND_QUOTE']
}
EUROSTOXX_MDX_CODE = 'STOX5E_X'

ezmdx.set_app(app_name='VEGA5')
ezmdx.prod.satis_login()
mtx_client = MdxClient('MSD', LOGIN_MDX, PASSWORD_MDX, use_prod_only=True)

today      = pd.Timestamp.today().normalize()
date_end   = today - BDay(1)
date_start = date_end - pd.DateOffset(years=5)

print(f'Période : {date_start.date()} → {date_end.date()}')

In [ ]:
# ============================================================
#  CHARGEMENT DES DONNÉES SX5E (cache des notebooks précédents)
# ============================================================
cache_path = cache_dir / 'sx5e_bergomi_5y_cache.pkl'

def get_market_data(mtx_client, asset_name, date_range, mdx_type):
    return mtx_client.get_market_data(mdx_type=mdx_type, code=asset_name, date=date_range)

def get_vol(asset_name, asset_type, date_start, date_end, mtx_client):
    all_bdays = pd.bdate_range(date_start, date_end).strftime('%Y-%m-%d').tolist()
    df = get_market_data(mtx_client, f'{asset_type}_{asset_name}', all_bdays, MDX_TYPES['volatility'])
    return df[['STRIKE', 'MATURITY', 'VOLATILITY', 'DATE']].copy()

def get_spot(mtx_client, asset_name, date_start, date_end):
    all_bdays = pd.bdate_range(date_start, date_end).strftime('%Y-%m-%d').tolist()
    for mdx_type in MDX_TYPES['spot']:
        try:
            return get_market_data(mtx_client, asset_name, all_bdays, mdx_type)
        except Exception:
            continue

if cache_path.exists():
    print('Cache trouvé, chargement...')
    with open(cache_path, 'rb') as f:
        cached = pickle.load(f)
    vols_raw  = cached['vols']
    spots_raw = cached['spots']
else:
    print('Fetching depuis MDX...')
    vols_raw  = get_vol(EUROSTOXX_MDX_CODE, 'I', date_start, date_end, mtx_client)
    spots_raw = get_spot(mtx_client, EUROSTOXX_MDX_CODE,
                         date_start - BDay(5), date_end + BDay(5))
    with open(cache_path, 'wb') as f:
        pickle.dump({'vols': vols_raw, 'spots': spots_raw}, f)

print(f'Données : vols={vols_raw.shape}, spots={spots_raw.shape}')

---
## Section 1 — Surface de volatilité implicite : définitions

### 1.1 Rappel Black-Scholes

Pour un call européen de strike $K$ et maturité $\tau = T - t$ :
$$C^{BS}(S_t, K, \tau, \sigma) = S_t N(d_1) - K e^{-r\tau} N(d_2) \tag{1}$$
$$d_1 = \frac{-\ln m + \tau(r + \sigma^2/2)}{\sigma\sqrt{\tau}}, \quad d_2 = d_1 - \sigma\sqrt{\tau}$$
où $m = K/S_t$ est la moneyness.

### 1.2 Volatilité implicite

La **volatilité implicite** $\sigma^{BS}_t(K,T)$ est l'unique valeur qui égalise le prix de marché et le prix BS :
$$C^{BS}(S_t, K, \tau, \sigma^{BS}_t(K,T)) = C^*_t(K,T) \tag{3}$$

La **surface de volatilité implicite** $I_t(m, \tau)$ est sa représentation en coordonnées relatives :
$$I_t(m, \tau) = \sigma^{BS}_t(m \cdot S(t),\; t+\tau)$$

### 1.3 Règles déterministes (insuffisantes)

- **Sticky moneyness :** $I_{t+\delta t}(m, \tau) = I_t(m, \tau)$ — surface figée en coordonnées relatives
- **Sticky strike :** $\sigma^{BS}_{t+\delta t}(K, T) = \sigma^{BS}_t(K, T)$ — surface figée en coordonnées absolues

Ces règles sont **déterministes** et ne capturent pas l'incertitude réelle de la surface.

In [ ]:
# ============================================================
#  UTILITAIRES BLACK-SCHOLES
# ============================================================
def bs_price(S, K, tau, sigma, r=0., q=0., option='call'):
    if tau <= 0 or sigma <= 0:
        return max(S-K,0.) if option=='call' else max(K-S,0.)
    m  = K / S
    d1 = (-np.log(m) + tau*(r-q+0.5*sigma**2)) / (sigma*np.sqrt(tau))
    d2 = d1 - sigma*np.sqrt(tau)
    disc = np.exp(-r*tau)
    if option=='call':
        return S*np.exp(-q*tau)*norm.cdf(d1) - K*disc*norm.cdf(d2)
    return K*disc*norm.cdf(-d2) - S*np.exp(-q*tau)*norm.cdf(-d1)

def implied_vol(S, K, tau, price, r=0., q=0., option='call'):
    try:
        return brentq(lambda v: bs_price(S,K,tau,v,r,q,option)-price,
                      1e-4, 5., xtol=1e-8)
    except Exception:
        return np.nan

def bs_vega(S, K, tau, sigma, r=0., q=0.):
    if tau <= 0 or sigma <= 0:
        return 0.
    d1 = (-np.log(K/S) + tau*(r-q+0.5*sigma**2)) / (sigma*np.sqrt(tau))
    return S * np.exp(-q*tau) * norm.pdf(d1) * np.sqrt(tau)

print('Utilitaires BS définis.')

In [ ]:
# ============================================================
#  PRÉPARATION DES DONNÉES BRUTES
# ============================================================
def prepare_raw_data(vols_raw, spots_raw, r=0., q=0.):
    vols = vols_raw.rename(columns={
        'STRIKE':'strike','MATURITY':'maturity_date',
        'VOLATILITY':'market_iv','DATE':'date'})
    for col in ['date','maturity_date']:
        vols[col] = pd.to_datetime(vols[col])
    vols['strike']    = pd.to_numeric(vols['strike'],    errors='coerce')
    vols['market_iv'] = pd.to_numeric(vols['market_iv'], errors='coerce')
    if vols['market_iv'].median() > 2:
        vols['market_iv'] /= 100.

    spots = spots_raw.copy()
    spots['date'] = pd.to_datetime(spots['DATE'])
    spot_col = [c for c in spots.columns if c != 'DATE'][0]
    spots['spot'] = pd.to_numeric(spots[spot_col], errors='coerce')

    df = vols.merge(spots[['date','spot']], on='date', how='left').dropna()
    df['bdays'] = [len(pd.bdate_range(d,m))-1 for d,m in zip(df['date'],df['maturity_date'])]
    df = df[df['bdays'] > 0]
    df['tau'] = df['bdays'] / 252.              # temps à maturité en années
    df['m']   = df['strike'] / df['spot']       # moneyness m = K/S

    # Filtres comme dans le papier : m ∈ [0.5, 1.5], τ ∈ [2/52, 1.5]
    df = df[(df['m'] >= 0.5) & (df['m'] <= 1.5)]
    df = df[(df['tau'] >= 2./52.) & (df['tau'] <= 1.5)]
    df = df[(df['market_iv'] > 0.01) & (df['market_iv'] < 2.0)]

    return df.sort_values(['date','tau','m']).reset_index(drop=True)

raw_data = prepare_raw_data(vols_raw, spots_raw)
print(f'Données filtrées : {raw_data.shape[0]:,} points')
print(f'Dates disponibles : {raw_data["date"].nunique()}')
print(f'Plage moneyness  : [{raw_data["m"].min():.3f}, {raw_data["m"].max():.3f}]')
print(f'Plage maturité   : [{raw_data["tau"].min():.3f}, {raw_data["tau"].max():.3f}] ans')
raw_data.head(6)

---
## Section 2 — Lissage non-paramétrique : estimateur de Nadaraya-Watson

### 2.1 Problème

Les prix de marché sont observés sur une grille **irrégulière et variable** de $(m_i, \tau_i)$. Pour construire une série temporelle de surfaces lisses, on a besoin d'interpoler sur une grille fixe.

### 2.2 Estimateur de Nadaraya-Watson (éq. 7)

$$\hat{I}_t(m, \tau) = \frac{\sum_{i=1}^n I_t(m_i, \tau_i)\, g(m-m_i, \tau-\tau_i)}{\sum_{i=1}^n g(m-m_i, \tau-\tau_i)} \tag{7}$$

avec un noyau gaussien :
$$g(x, y) = \frac{1}{2\pi} \exp\!\left(-\frac{x^2}{2h_1}\right)\exp\!\left(-\frac{y^2}{2h_2}\right)$$

**Paramètres de bande passante :** $h_1$ (moneyness), $h_2$ (maturité). Trop petits → surface trop rugueuse ; trop grands → sur-lissage.

### 2.3 Grille fixe

On travaille en log-volatilité : $X_t(m, \tau) = \ln I_t(m, \tau)$ — ce qui assure la positivité et se prête mieux à la décomposition KL.

In [ ]:
# ============================================================
#  GRILLE FIXE ET ESTIMATEUR DE NADARAYA-WATSON
# ============================================================

# Grille fixe (m, τ) — identique pour toutes les dates
M_GRID   = np.array([0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20])
TAU_GRID = np.array([1./12., 2./12., 3./12., 6./12., 9./12., 12./12.])

M_MESH, TAU_MESH = np.meshgrid(M_GRID, TAU_GRID, indexing='ij')  # shape (nM, nT)
N_M, N_T = len(M_GRID), len(TAU_GRID)

# Bandes passantes (cross-validation suggère h1 ≈ 0.1, h2 ≈ 0.1 en années)
H1 = 0.10  # bandwidth moneyness
H2 = 0.15  # bandwidth maturité

def nadaraya_watson(m_obs, tau_obs, iv_obs, m_grid, tau_grid, h1, h2):
    """
    Estimateur de Nadaraya-Watson — éq. (7) de Cont & da Fonseca.

    Paramètres
    ----------
    m_obs, tau_obs, iv_obs : observations brutes du jour
    m_grid, tau_grid       : grille fixe 1D
    h1, h2                 : bandes passantes

    Retourne
    --------
    iv_smooth : surface lissée (shape nM × nT)
    """
    nM, nT = len(m_grid), len(tau_grid)
    iv_smooth = np.full((nM, nT), np.nan)

    for i, m in enumerate(m_grid):
        for j, tau in enumerate(tau_grid):
            # Poids du noyau gaussien
            dm   = m_obs - m
            dtau = tau_obs - tau
            K    = np.exp(-dm**2 / (2*h1**2) - dtau**2 / (2*h2**2))
            denom = K.sum()
            if denom < 1e-15:
                continue
            iv_smooth[i, j] = (K * iv_obs).sum() / denom

    return iv_smooth


def build_surface_series(raw_data, m_grid, tau_grid, h1, h2,
                          min_obs=10):
    """
    Construit la série temporelle de surfaces lissées.
    Retourne :
      surfaces_iv  : dict {date: array (nM, nT)} — vol implicite lissée
      surfaces_log : dict {date: array (nM, nT)} — log(vol implicite)
      spots        : dict {date: float} — prix spot
    """
    surfaces_iv  = {}
    surfaces_log = {}
    spot_series  = {}

    dates = raw_data['date'].unique()

    for date in sorted(dates):
        day = raw_data[raw_data['date'] == date]
        if len(day) < min_obs:
            continue

        m_obs   = day['m'].values
        tau_obs = day['tau'].values
        iv_obs  = day['market_iv'].values

        surf = nadaraya_watson(m_obs, tau_obs, iv_obs, m_grid, tau_grid, h1, h2)

        # Vérifier qu'il n'y a pas trop de NaN
        if np.isnan(surf).sum() > 0.3 * surf.size:
            continue

        # Remplir les NaN par interpolation bilinéaire simple
        surf_filled = pd.DataFrame(surf).interpolate(axis=0).interpolate(axis=1).values

        surfaces_iv[date]  = np.maximum(surf_filled, 0.01)
        surfaces_log[date] = np.log(np.maximum(surf_filled, 0.01))
        spot_series[date]  = day['spot'].iloc[0]

    return surfaces_iv, surfaces_log, spot_series


# Construction (avec cache)
surf_cache = cache_dir / 'cont_surfaces.pkl'

if surf_cache.exists():
    print('Cache surfaces trouvé, chargement...')
    with open(surf_cache, 'rb') as f:
        cached_surf = pickle.load(f)
    surfaces_iv  = cached_surf['iv']
    surfaces_log = cached_surf['log']
    spot_series  = cached_surf['spots']
else:
    print('Construction des surfaces lissées (peut prendre quelques minutes)...')
    surfaces_iv, surfaces_log, spot_series = build_surface_series(
        raw_data, M_GRID, TAU_GRID, H1, H2
    )
    with open(surf_cache, 'wb') as f:
        pickle.dump({'iv': surfaces_iv, 'log': surfaces_log, 'spots': spot_series}, f)
    print('Surfaces sauvegardées.')

dates_sorted = sorted(surfaces_iv.keys())
N_dates = len(dates_sorted)
print(f'{N_dates} surfaces construites sur la grille {N_M}×{N_T}')

---
## Section 3 — Analyse de variance : profil moyen et écart-type

### 3.1 Profil moyen

La surface moyenne $\bar{I}(m,\tau)$ montre les deux caractéristiques empiriques bien connues :
1. **Skew** : vol décroissante avec la moneyness (protection put > call)
2. **Structure par terme** : vol généralement décroissante avec la maturité

### 3.2 Écart-type quotidien

La **déviation standard des variations log-quotidiennes** $X_t(m,\tau) = \ln I_t(m,\tau) - \ln I_{t-1}(m,\tau)$ quantifie l'intensité des fluctuations de la surface — c'est le test du modèle sticky moneyness.

In [ ]:
# ============================================================
#  PROFIL MOYEN ET ÉCART-TYPE — FIGURES 4, 5, 6
# ============================================================

# Empiler toutes les surfaces en un tableau 3D : (N_dates, N_M, N_T)
surf_stack_iv  = np.array([surfaces_iv[d]  for d in dates_sorted])   # vols
surf_stack_log = np.array([surfaces_log[d] for d in dates_sorted])   # log-vols

# Variations log-quotidiennes : X_t = ln(I_t) - ln(I_{t-1})
log_variations = np.diff(surf_stack_log, axis=0)  # (N-1, N_M, N_T)

# Statistiques
mean_iv    = surf_stack_iv.mean(axis=0)         # (N_M, N_T)
mean_log   = surf_stack_log.mean(axis=0)
std_logvar = log_variations.std(axis=0)         # (N_M, N_T)

print(f'Variations log-quotidiennes : {log_variations.shape}')
print(f'Std moyenne : {std_logvar.mean()*100:.2f}%')
print(f'Std max     : {std_logvar.max()*100:.2f}%')

In [ ]:
# ============================================================
#  FIGURES 4, 5, 6 — RÉPLIQUE CONT & DA FONSECA
# ============================================================
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Labels de la grille
m_labels   = [f'{m:.0%}' for m in M_GRID]
tau_labels = ['1M', '2M', '3M', '6M', '9M', '12M']

# --- Figure 4 : Surface moyenne de vol implicite ---
ax1 = fig.add_subplot(gs[0, 0], projection='3d')
ax1.plot_surface(M_MESH, TAU_MESH*12, mean_iv*100,
                 cmap='viridis', alpha=0.85, edgecolor='none')
ax1.set_xlabel('Moneyness m')
ax1.set_ylabel('Maturité (mois)')
ax1.set_zlabel('Vol impl. (%)')
ax1.set_title('Figure 4 — Surface moyenne\nSX5E')
ax1.view_init(30, -60)

# --- Figure 5 : Surface moyenne log-vol ---
ax2 = fig.add_subplot(gs[0, 1], projection='3d')
ax2.plot_surface(M_MESH, TAU_MESH*12, mean_log,
                 cmap='plasma', alpha=0.85, edgecolor='none')
ax2.set_xlabel('Moneyness m')
ax2.set_ylabel('Maturité (mois)')
ax2.set_zlabel('log(Vol)')
ax2.set_title('Figure 5 — Surface log-vol moyenne')
ax2.view_init(30, -60)

# --- Figure 6 : Écart-type des variations log-quotidiennes ---
ax3 = fig.add_subplot(gs[0, 2], projection='3d')
ax3.plot_surface(M_MESH, TAU_MESH*12, std_logvar*100,
                 cmap='hot', alpha=0.85, edgecolor='none')
ax3.set_xlabel('Moneyness m')
ax3.set_ylabel('Maturité (mois)')
ax3.set_zlabel('Std log-var (%)')
ax3.set_title('Figure 6 — Écart-type quotidien\n(test sticky moneyness)')
ax3.view_init(30, -60)

# --- Heatmaps 2D (plus lisibles) ---
for ax_idx, (data, title, cmap) in enumerate([
    (mean_iv*100,     'Surface moyenne vol %',          'viridis'),
    (std_logvar*100,  'Std quotidienne log-vol (%)',      'hot'),
    (mean_iv/mean_iv[:,4:5], 'Vol relative à ATM', 'RdBu_r'),
]):
    ax = fig.add_subplot(gs[1, ax_idx])
    im = ax.imshow(data.T, aspect='auto', origin='lower',
                   cmap=cmap, extent=[0.8, 1.2, 1, 12])
    ax.set_xlabel('Moneyness m')
    ax.set_ylabel('Maturité (mois)')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Section 3 — Profil moyen, log-vol et écart-type quotidien (SX5E)\n'
             'Réplique Cont & da Fonseca (2002) — Figures 4, 5, 6',
             fontweight='bold', fontsize=13)
plt.savefig('cont_figs_4_5_6.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nObservation clé — test sticky moneyness :')
print(f'  Std quotidienne max = {std_logvar.max()*100:.2f}%  '
      f'≈ {std_logvar.max()/mean_iv.mean()*100:.1f}% de la vol moyenne')
print("  → Le modèle 'fixed smile' est clairement rejeté")

---
## Section 4 — Décomposition de Karhunen-Loève (KL) : théorie

### 4.1 Cadre mathématique

Soit $U_t(m, \tau)$ le champ aléatoire des **variations log-quotidiennes** de la surface. On le modélise comme un champ aléatoire stationnaire sur $A = [m_{\min}, m_{\max}] \times [\tau_{\min}, \tau_{\max}]$.

### 4.2 Noyau de covariance

$$K(x_1, x_2) = \text{cov}(U(x_1), U(x_2)), \qquad x_1, x_2 \in A \tag{9}$$

L'opérateur intégral associé :
$$\mathcal{K}u(x) = \int_A K(x,y)\,u(y)\,dy \tag{10}$$

### 4.3 Modes propres

Les vecteurs propres $f_n$ de $\mathcal{K}$ satisfont l'équation de Fredholm :
$$\int K(x,y)\,f_n(y)\,dy = \nu_n^2 f_n(x), \qquad f_j, f_k = \delta_{j,k} \tag{11}$$

### 4.4 Décomposition KL (éq. 13)

$$\boxed{U(\omega, \cdot) = \sum_n U_n(\omega)\,f_n(\cdot), \qquad U_n(\omega) = \langle U(\omega,\cdot), f_n \rangle}$$

Les $U_n$ sont **décorrélés** : $\mathbb{E}[U_j U_k] = \nu_j^2 \delta_{jk}$.

**La surface de vol implicite s'écrit alors (éq. 21) :**
$$I_t(m, \tau) = I_0(m, \tau) \exp\!\left(\sum_k x_k(t) f_k(m, \tau)\right) \tag{21}$$

---
## Section 5 — Implémentation numérique : problème aux valeurs propres généralisé

### 5.1 Méthode de Galerkin

On projette les fonctions propres sur une base finie $(h_j)$ :
$$f_i(m,\tau) = \sum_{n=1}^N a_{ij} h_j(m,\tau) \tag{14}$$

La condition de Galerkin $\langle \varepsilon_N, h_j \rangle = 0$ donne le problème généralisé :
$$CA = DBA \tag{19}$$

où :
- $B_{ij} = \langle h_i, h_j \rangle$ — matrice de Gram
- $C_{ij} = \int\int h_i(x) K(x,x') h_j(x')\, dx\, dx'$ — matrice de covariance projetée
- $D = \text{diag}(\nu_k^2)$ — valeurs propres

### 5.2 Implémentation sur grille discrète

Sur une grille finie $(m_i, \tau_j)$, les intégrales se remplacent par des sommes pondérées et le problème de Galerkin devient un problème aux valeurs propres ordinaire sur la matrice de covariance empirique.

In [ ]:
# ============================================================
#  DÉCOMPOSITION DE KARHUNEN-LOÈVE — IMPLÉMENTATION
# ============================================================
# Reshape : chaque surface devient un vecteur de taille N_M*N_T
N_pts  = N_M * N_T
N_time = log_variations.shape[0]  # nombre de variations journalières

# Matrice des observations : (N_time, N_pts)
X_mat = log_variations.reshape(N_time, N_pts)  # chaque ligne = un jour

# Centrage (on travaille déjà sur les variations, la moyenne est ~0)
X_mean = X_mat.mean(axis=0)
X_centered = X_mat - X_mean

# Matrice de covariance empirique : (N_pts, N_pts)
# K(x1,x2) = cov(U(x1), U(x2))
cov_matrix = X_centered.T @ X_centered / (N_time - 1)
print(f'Matrice de covariance : {cov_matrix.shape}')

# Décomposition en valeurs propres (par valeur décroissante)
# eigh renvoie les valeurs propres en ordre croissant → on inverse
eigenvalues, eigenvectors = eigh(cov_matrix)
eigenvalues  = eigenvalues[::-1]
eigenvectors = eigenvectors[:, ::-1]

# Variance expliquée par chaque mode
total_var   = eigenvalues.sum()
var_expl    = eigenvalues / total_var * 100
var_cum     = np.cumsum(var_expl)

print(f'\nVariance expliquée par mode :')
for k in range(min(6, len(eigenvalues))):
    print(f'  Mode {k+1} : {var_expl[k]:.2f}%  (cumulé : {var_cum[k]:.2f}%)')

# Modes propres en format surface (N_M, N_T)
N_MODES = 3
eigenmodes = [eigenvectors[:, k].reshape(N_M, N_T) for k in range(N_MODES)]

# Projections (composantes principales) : x_k(t)
projections = {}
for k in range(N_MODES):
    projections[k] = X_centered @ eigenvectors[:, k]  # shape (N_time,)

# Série temporelle associée aux dates
dates_var = dates_sorted[1:]  # une de moins car différences
proj_df = pd.DataFrame(
    {f'x{k+1}': projections[k] for k in range(N_MODES)},
    index=pd.to_datetime(dates_var)
)

In [ ]:
# ============================================================
#  FIGURE 7 — VALEURS PROPRES (Réplique)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gauche : valeurs propres (variance par mode)
n_show = min(15, len(eigenvalues))
axes[0].bar(range(1, n_show+1), var_expl[:n_show],
            color='steelblue', alpha=0.8, edgecolor='k', lw=0.5)
axes[0].set_xlabel('Rang du mode')
axes[0].set_ylabel('Variance expliquée (%)')
axes[0].set_title('Figure 7 — Réplique Cont & da Fonseca\nValeurs propres (variance par mode)')
for k in range(3):
    axes[0].text(k+1, var_expl[k]+0.3, f'{var_expl[k]:.1f}%', ha='center', fontsize=9)

# Droite : variance cumulée
axes[1].plot(range(1, n_show+1), var_cum[:n_show], 'o-', color='firebrick', lw=2, ms=7)
axes[1].axhline(95., ls='--', color='gray', lw=1.5, label='95%')
axes[1].axhline(98., ls=':', color='orange', lw=1.5, label='98%')
axes[1].set_xlabel('Nombre de modes')
axes[1].set_ylabel('Variance cumulée (%)')
axes[1].set_title('Variance cumulée')
axes[1].legend()

n_95 = np.argmax(var_cum >= 95.) + 1
n_98 = np.argmax(var_cum >= 98.) + 1
axes[1].axvline(n_95, ls='--', color='gray', alpha=0.7)
axes[1].axvline(n_98, ls=':', color='orange', alpha=0.7)

plt.suptitle('Section 5 — Décroissance rapide des valeurs propres', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n{n_95} modes pour 95% de la variance  (Cont & da Fonseca : 2-3)')
print(f'{n_98} modes pour 98% de la variance')
print(f'Mode 1 seul : {var_expl[0]:.1f}%  (papier SP500 : ~80%, FTSE : ~96%)')

---
## Section 6 — Modes propres : interprétation niveau, skew, butterfly

### 6.1 Premier mode : **niveau** (level factor)

Tous les composantes sont **positives** → un choc positif augmente uniformément toutes les volatilités implicites. C'est le facteur de **niveau global** de la surface. Il correspond à l'**effet levier** : fortement corrélé négativement avec les rendements du sous-jacent.

### 6.2 Deuxième mode : **skew** (slope factor)

Change de signe à la moneyness : **négatif pour $m < 1$** (puts OTM), **positif pour $m > 1$** (calls OTM). Un choc positif incline le smile → modifie le **skewness** de la distribution risque-neutre.

### 6.3 Troisième mode : **convexité** (butterfly factor)

Forme en U : modifie la **courbure** du smile → impact sur le **kurtosis** de la distribution risque-neutre.

In [ ]:
# ============================================================
#  FIGURES 8, 11, 14 — MODES PROPRES : RÉPLIQUE
# ============================================================
mode_titles = [
    f'Figure 8 — Mode 1 : NIVEAU\n({var_expl[0]:.1f}% variance)',
    f'Figure 11 — Mode 2 : SKEW\n({var_expl[1]:.1f}% variance)',
    f'Figure 14 — Mode 3 : BUTTERFLY\n({var_expl[2]:.1f}% variance)',
]
mode_interp = [
    'Tous composantes positifs → choc uniforme\nFortement corrélé avec rendements sous-jacent',
    'Change de signe à ATM → modifie le skewness\nPeu corrélé avec le sous-jacent',
    'Forme en U → modifie la convexité\nFaible impact relatif'
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

cmaps = ['viridis', 'RdBu_r', 'RdBu_r']

for k, (mode, title, interp, cmap) in enumerate(
        zip(eigenmodes, mode_titles, mode_interp, cmaps)):

    # Top : surface 3D du mode
    ax_3d = fig.add_subplot(2, 3, k+1, projection='3d')
    surf_max = np.abs(mode).max()
    ax_3d.plot_surface(M_MESH, TAU_MESH*12, mode,
                       cmap=cmap, alpha=0.85, edgecolor='none',
                       vmin=-surf_max, vmax=surf_max)
    ax_3d.set_xlabel('m', fontsize=8)
    ax_3d.set_ylabel('τ (mois)', fontsize=8)
    ax_3d.set_zlabel(f'$f_{k+1}$', fontsize=8)
    ax_3d.set_title(title, fontsize=10)
    ax_3d.view_init(25, -50)

    # Bottom : coupe à τ fixe (pour les 3 maturités principales)
    ax_2d = axes[1, k]
    for j_tau, tau_val in [(0, TAU_GRID[0]), (2, TAU_GRID[2]), (5, TAU_GRID[5])]:
        ax_2d.plot(M_GRID, mode[:, j_tau], lw=2,
                   label=f'τ={tau_val*12:.0f}M')
    ax_2d.axhline(0., color='k', lw=0.8, ls='--')
    ax_2d.axvline(1., color='gray', lw=0.8, ls=':')
    ax_2d.set_xlabel('Moneyness m')
    ax_2d.set_ylabel(f'$f_{k+1}(m, \\tau)$')
    ax_2d.set_title(f'Coupe en moneyness — Mode {k+1}\n{interp}', fontsize=9)
    ax_2d.legend(fontsize=8)

plt.suptitle('Section 6 — Trois premiers modes propres : Niveau, Skew, Butterfly\n'
             'Réplique Cont & da Fonseca (2002) — Figures 8, 11, 14',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('cont_eigenmodes.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7 — Dynamique des processus de composantes principales

### 7.1 Propriétés empiriques

Cont & da Fonseca rapportent pour SP500 (Table 1) et FTSE (Table 2) :

| Mode | Variance (%) | Kurtosis | Mean-reversion (jours) | Corr. sous-jacent |
|---|---|---|---|---|
| 1 (niveau) | 94 (SP500), 96 (FTSE) | 6.4 | 28 | −0.66 |
| 2 (skew) | 3 | 7.9 | 12.6 | ≈ 0 |
| 3 (butterfly) | 0.8 | 7.8 | 22 | 0.27 |

**Observations clés :**
- Forte autocorrélation et mean-reversion à une échelle de temps ≈ 1 mois
- Structure AR(1)/OU bien approximée
- Distributions non-gaussiennes (excès de kurtosis)

In [ ]:
# ============================================================
#  FIGURES 9, 10, 12, 13 — PROJECTIONS ET AUTOCORRÉLATIONS
# ============================================================
fig, axes = plt.subplots(3, 3, figsize=(18, 14))

mode_colors = ['steelblue', 'firebrick', 'forestgreen']
mode_labels = ['Niveau (Mode 1)', 'Skew (Mode 2)', 'Butterfly (Mode 3)']

for k in range(N_MODES):
    x_series = proj_df[f'x{k+1}']

    # Colonne 0 : série temporelle de la projection
    axes[k, 0].plot(proj_df.index, x_series, color=mode_colors[k], lw=1.2, alpha=0.85)
    axes[k, 0].axhline(0., color='k', lw=0.7, ls='--')
    axes[k, 0].set_title(f'{mode_labels[k]}\nProjection $x_{k+1}(t)$', fontsize=10)
    axes[k, 0].set_ylabel(f'$x_{k+1}(t)$')
    axes[k, 0].xaxis.set_tick_params(rotation=30, labelsize=8)

    # Colonne 1 : autocorrélation
    nlags = 60
    try:
        acf_vals  = acf(x_series.dropna(),  nlags=nlags, fft=True)
        pacf_vals = pacf(x_series.dropna(), nlags=nlags)
    except Exception:
        acf_vals  = np.ones(nlags+1)
        pacf_vals = np.ones(nlags+1)

    lags = np.arange(nlags+1)
    axes[k, 1].bar(lags, acf_vals, color=mode_colors[k], alpha=0.7, width=0.8)
    # Décroissance exponentielle théorique AR(1)
    ar1_coeff = acf_vals[1] if len(acf_vals) > 1 else 0.9
    mr_time = -1. / np.log(max(abs(ar1_coeff), 1e-5))
    exp_decay = ar1_coeff**lags
    axes[k, 1].plot(lags, exp_decay, 'k--', lw=1.5,
                     label=f'e^(-t/{mr_time:.0f}d)')
    axes[k, 1].axhline(0., color='k', lw=0.7)
    axes[k, 1].set_title(f'ACF de $x_{k+1}$\n(Mean-reversion ≈ {mr_time:.0f} jours)')
    axes[k, 1].set_xlabel('Lag (jours)')
    axes[k, 1].legend(fontsize=8)

    # Colonne 2 : PACF + distribution
    axes[k, 2].bar(lags[:20], pacf_vals[:20], color=mode_colors[k], alpha=0.7, width=0.8)
    axes[k, 2].axhline(0., color='k', lw=0.7)
    axes[k, 2].axhline( 1.96/np.sqrt(len(x_series)), 'k', ls=':', alpha=0.5)
    axes[k, 2].axhline(-1.96/np.sqrt(len(x_series)), 'k', ls=':', alpha=0.5)
    axes[k, 2].set_title(f'PACF de $x_{k+1}$ (lag max 20)')
    axes[k, 2].set_xlabel('Lag (jours)')

plt.suptitle('Section 7 — Dynamique des composantes principales : AR(1)/OU\n'
             'Réplique Cont & da Fonseca — Figures 9, 10, 12, 13',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('cont_projections.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
#  TABLE 1 & 2 — STATISTIQUES DES COMPOSANTES PRINCIPALES
# ============================================================
spot_ser = pd.Series(spot_series)
spot_ser.index = pd.to_datetime(spot_ser.index)
spot_ret = np.log(spot_ser).diff().dropna()

stats_rows = []
for k in range(N_MODES):
    x_k   = proj_df[f'x{k+1}'].dropna()
    dx_k  = x_k.diff().dropna()

    # AR(1) estimation
    try:
        ar1_model = AutoReg(x_k, lags=1, old_names=False).fit()
        ar1_coef  = ar1_model.params.iloc[1]
        mr_days   = -1. / np.log(max(abs(ar1_coef), 1e-6))
    except Exception:
        ar1_coef = 0.95
        mr_days  = 20.

    # Corrélation avec le sous-jacent
    common_idx = dx_k.index.intersection(spot_ret.index)
    if len(common_idx) > 10:
        corr_underlying = np.corrcoef(
            dx_k.loc[common_idx].values,
            spot_ret.loc[common_idx].values
        )[0, 1]
    else:
        corr_underlying = np.nan

    stats_rows.append({
        'Mode': f'{k+1} — {["Niveau","Skew","Butterfly"][k]}',
        'Std journalière': f'{x_k.std():.4f}',
        'Variance (%)': f'{var_expl[k]:.2f}',
        'Kurtosis': f'{scipy_kurtosis(x_k, fisher=False):.2f}',
        'Skewness': f'{scipy_skew(x_k):.2f}',
        'Mean-reversion (jours)': f'{mr_days:.1f}',
        'Corr sous-jacent': f'{corr_underlying:.3f}',
    })

df_stats = pd.DataFrame(stats_rows).set_index('Mode')

print('Table 1 — Statistiques des composantes principales (SX5E, 5 ans) :')
print('(Réplique Tables 1 & 2 de Cont & da Fonseca 2002)\n')
print(df_stats.to_string())

print('\nRéférence papier (SP500, 2000-2001) :')
print('  Mode 1 : Std=0.10, Var=94%, Kurt=6.4, MR=28j, Corr=-0.66')
print('  Mode 2 : Std=0.09, Var=3%,  Kurt=7.9, MR=12.6j, Corr≈0')
print('  Mode 3 : Std=0.05, Var=0.8%, Kurt=7.8, MR=22j, Corr=0.27')

---
## Section 8 — Modèle à facteurs : Ornstein-Uhlenbeck & calibration AR(1)

### 8.1 Modèle factoriel (éq. 22–24)

$$\ln I_t(m, \tau) = X_t(m, \tau) = X_0(m, \tau) + \sum_{k=1}^d x_k(t)\,f_k(m, \tau) \tag{22}$$

où les composantes $x_k(t)$ suivent des processus d'**Ornstein-Uhlenbeck indépendants** :
$$dx_k(t) = -\lambda_k(x_k(t) - \bar{x}_k)\,dt + \gamma_k\,dZ_k(t) \tag{24}$$

### 8.2 Relations aux statistiques empiriques (éq. 25–27)

$$\bar{x}_k = \langle X - X_0, f_k \rangle \tag{25}$$
$$\nu_k^2 = \frac{\gamma_k^2}{2\lambda_k} \tag{26}$$
$$\text{corr}(x_k(t), x_k(t+s)) = e^{-\lambda_k s} \tag{27}$$

### 8.3 Discrétisation AR(1) (éq. 28–29)

$$x_k(t+1) = e^{-\lambda_k}\,x_k(t) + (1 - e^{-\lambda_k})\bar{x}_k + \sigma_k\,\varepsilon_k(t) \tag{28}$$
$$\sigma_k = \gamma_k\sqrt{\frac{1-e^{-2\lambda_k}}{2\lambda_k}} \tag{29}$$

In [ ]:
# ============================================================
#  CALIBRATION DU MODÈLE OU/AR(1) — TABLE 3
# ============================================================
def calibrate_OU(x_series):
    """
    Calibre le processus OU : dx = -λ(x - x̄)dt + γ dZ
    via régression AR(1) sur observations journalières.

    Retourne : lambda (par jour), x_bar, gamma (par jour^{1/2})
    """
    x = x_series.dropna().values
    if len(x) < 30:
        return np.nan, 0., np.nan

    # Régression AR(1) : x(t+1) = a + b*x(t) + ε
    x_lag  = x[:-1]
    x_next = x[1:]
    b, a   = np.polyfit(x_lag, x_next, 1)

    # Relations OU : b = e^{-λ}, a = (1-b)*x̄
    b       = np.clip(b, 0.001, 0.9999)
    lam     = -np.log(b)         # λ en jours^{-1}
    x_bar   = a / (1 - b)       # valeur moyenne long terme
    resid   = x_next - (a + b*x_lag)
    sigma   = resid.std()        # bruit AR(1) journalier
    gamma   = sigma * np.sqrt(2*lam / (1 - np.exp(-2*lam)))

    return lam, x_bar, gamma


ou_params = {}
for k in range(N_MODES):
    lam, x_bar, gamma = calibrate_OU(proj_df[f'x{k+1}'])
    mr_days = 1. / lam if lam > 0 else np.nan
    ou_params[k] = {'lambda': lam, 'x_bar': x_bar, 'gamma': gamma,
                     'mr_days': mr_days}

print('Table 3 — Calibration du modèle OU/AR(1) (éq. 28) :')
print('(Réplique Table 3 de Cont & da Fonseca 2002)\n')
df_ou = pd.DataFrame(ou_params).T
df_ou.index = [f'Mode {k+1}' for k in range(N_MODES)]
df_ou['lambda'] = df_ou['lambda'].map('{:.4f}/j'.format)
df_ou['mr_days'] = df_ou['mr_days'].map('{:.1f} j'.format)
df_ou['gamma']  = df_ou['gamma'].map('{:.4f}/j^0.5'.format)
df_ou['x_bar']  = df_ou['x_bar'].map('{:.4f}'.format)
print(df_ou.to_string())

print('\nRéférence papier (SP500) :')
print('  λ1 = 0.035, λ2 = 0.080, λ3 = 0.045')
print('  (FTSE : λ1 = 0.019, λ2 = 0.015, λ3 = 0.012)')

In [ ]:
# ============================================================
#  SIMULATION DU MODÈLE FACTORIEL
# ============================================================
def simulate_factor_model(ou_params, eigenmodes, surface_0_log,
                           N_sim=252, n_paths=5, seed=42, n_modes=3):
    """
    Simule la dynamique de la surface de vol via le modèle OU à n_modes facteurs.

    Retourne :
      x_paths    : (n_paths, N_sim, n_modes) — projections simulées
      surf_paths : (n_paths, N_sim, N_M, N_T) — surfaces simulées (IV)
    """
    rng = np.random.default_rng(seed)

    # Initialisation
    x0 = np.array([ou_params[k]['x_bar'] for k in range(n_modes)])

    x_paths    = np.zeros((n_paths, N_sim, n_modes))
    surf_paths = np.zeros((n_paths, N_sim, N_M, N_T))

    for p in range(n_paths):
        x = x0.copy()
        for t in range(N_sim):
            x_paths[p, t] = x

            # Log-vol simulée
            log_surf = surface_0_log.copy()
            for k in range(n_modes):
                log_surf += x[k] * eigenmodes[k]
            surf_paths[p, t] = np.exp(log_surf)

            # Avancer x via AR(1)
            Z = rng.standard_normal(n_modes)
            for k in range(n_modes):
                lam   = ou_params[k]['lambda']
                xbar  = ou_params[k]['x_bar']
                gamma = ou_params[k]['gamma']
                sigma_dt = gamma * np.sqrt((1 - np.exp(-2*lam)) / (2*lam))
                x[k]  = np.exp(-lam)*x[k] + (1-np.exp(-lam))*xbar + sigma_dt*Z[k]

    return x_paths, surf_paths


# Surface initiale : surface la plus récente
surf0_log = surfaces_log[dates_sorted[-1]]

x_paths, surf_paths = simulate_factor_model(
    ou_params, eigenmodes, surf0_log,
    N_sim=252, n_paths=5, seed=42
)

print(f'Simulation : {x_paths.shape[0]} chemins × {x_paths.shape[1]} jours')

# Visualisation : vol ATM simulée
atm_idx = np.argmin(np.abs(M_GRID - 1.0))
tau3m_idx = np.argmin(np.abs(TAU_GRID - 0.25))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for p in range(x_paths.shape[0]):
    atm_vol = surf_paths[p, :, atm_idx, tau3m_idx]
    axes[0].plot(atm_vol*100, alpha=0.7, lw=1.5)

axes[0].set_xlabel('Jours simulés')
axes[0].set_ylabel('Vol ATM 3M (%)')
axes[0].set_title('Chemins simulés — Vol ATM 3M\n(modèle factoriel 3-OU)')

for p in range(x_paths.shape[0]):
    axes[1].plot(x_paths[p, :, 0], alpha=0.7, lw=1.5)

axes[1].set_xlabel('Jours simulés')
axes[1].set_ylabel('$x_1(t)$ (Mode 1 — Niveau)')
axes[1].set_title('Projection sur le 1er mode\n(mean-reverting OU)')

plt.suptitle('Section 8 — Simulation du modèle factoriel OU', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 9 — Corrélation avec le sous-jacent (effet levier)

### 9.1 Résultats clés de Cont & da Fonseca

- **Mode 1 (niveau) :** corrélation de **−66%** avec les rendements du SP500 → **effet levier**
- **Mode 2 (skew) :** corrélation ≈ **0%** → les mouvements relatifs du smile sont décorrélés du sous-jacent
- **Mode 3 (butterfly) :** faible corrélation positive

**Implication cruciale :** Le risque Vega **ne peut pas** être réduit au risque Delta. Les modèles à un facteur ne sont pas suffisants.

In [ ]:
# ============================================================
#  CORRÉLATION COMPOSANTES PRINCIPALES / SOUS-JACENT
# ============================================================
# Séries de rendements
spot_ret_daily = np.log(spot_ser).diff().dropna()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for k in range(N_MODES):
    dx_k = proj_df[f'x{k+1}'].diff().dropna()
    common_idx = dx_k.index.intersection(spot_ret_daily.index)

    if len(common_idx) < 20:
        continue

    dx_arr   = dx_k.loc[common_idx].values
    ret_arr  = spot_ret_daily.loc[common_idx].values
    corr_val = np.corrcoef(dx_arr, ret_arr)[0, 1]

    # Haut : scatter plot
    axes[0, k].scatter(ret_arr*100, dx_arr, alpha=0.2, s=8,
                        color=mode_colors[k])
    # Régression
    from scipy.stats import linregress
    slope, intercept, _, _, _ = linregress(ret_arr, dx_arr)
    x_line = np.linspace(ret_arr.min(), ret_arr.max(), 100)
    axes[0, k].plot(x_line*100, (slope*x_line+intercept), 'k-', lw=2)
    axes[0, k].set_xlabel('Rendement sous-jacent (%)')
    axes[0, k].set_ylabel(f'$\\Delta x_{k+1}$')
    axes[0, k].set_title(f'Mode {k+1} — {mode_labels[k]}\nCorr = {corr_val:.3f}')

    # Bas : rolling correlation 63j
    roll_corr = pd.Series(dx_arr, index=common_idx).rolling(63).corr(
        pd.Series(ret_arr, index=common_idx)
    )
    axes[1, k].plot(roll_corr.index, roll_corr.values,
                    color=mode_colors[k], lw=1.5, alpha=0.85)
    axes[1, k].axhline(corr_val, ls='--', color='k', lw=1.5,
                        label=f'Moyenne = {corr_val:.3f}')
    axes[1, k].axhline(0., color='gray', lw=0.8)
    axes[1, k].set_ylabel('Corrélation (63j)')
    axes[1, k].set_title(f'Corrélation glissante 3M — Mode {k+1}')
    axes[1, k].legend(fontsize=8)
    axes[1, k].xaxis.set_tick_params(rotation=30, labelsize=8)

plt.suptitle('Section 9 — Corrélation modes propres / sous-jacent\n'
             'Effet levier (mode 1) vs décorrélation (mode 2)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Résumé
print('\nCorr modes propres / sous-jacent :')
for k in range(N_MODES):
    dx_k = proj_df[f'x{k+1}'].diff().dropna()
    common_idx = dx_k.index.intersection(spot_ret_daily.index)
    if len(common_idx) > 20:
        c = np.corrcoef(dx_k.loc[common_idx], spot_ret_daily.loc[common_idx])[0,1]
        print(f'  Mode {k+1} : corr = {c:.3f}  '
              f'(Cont&da Fonseca SP500: {[-0.66, 0.00, 0.27][k]})')

---
## Section 10 — Modèle factoriel complet & simulation de scénarios

### 10.1 Reconstruction et qualité d'ajustement

On vérifie que la reconstruction à 3 modes explique bien les données :
$$\hat{I}_t(m,\tau) = I_0(m,\tau)\exp\!\left(\sum_{k=1}^3 x_k(t)\,f_k(m,\tau)\right)$$

### 10.2 Règle sticky moneyness améliorée

Le modèle factoriel **étend** la règle sticky moneyness ($x_k = 0$) en permettant aux facteurs d'évoluer stochastiquement. C'est la contribution principale pour les praticiens.

In [ ]:
# ============================================================
#  RECONSTRUCTION ET QUALITÉ D'AJUSTEMENT
# ============================================================
def reconstruct_surface(projections_k, eigenmodes, surface_0_log, k_max=3):
    """
    Reconstruit la surface log-vol à partir des k_max premiers modes.
    projections_k : array (N_time, k_max) — projections journalières
    """
    N_time = projections_k.shape[0]
    log_surf_recon = np.zeros((N_time, N_M, N_T))

    for t in range(N_time):
        log_s = surface_0_log.copy()
        for k in range(k_max):
            log_s += projections_k[t, k] * eigenmodes[k]
        log_surf_recon[t] = log_s

    return log_surf_recon  # (N_time, N_M, N_T)


# Matrice des projections
proj_matrix = np.column_stack([projections[k] for k in range(N_MODES)])

# Reconstruction 1, 2, 3 modes
errors = {}
for n_k in [1, 2, 3]:
    recon = reconstruct_surface(proj_matrix, eigenmodes, surf0_log, k_max=n_k)
    actual = log_variations + surf0_log[None, :, :]  # approximation
    err = np.abs(recon - surf_stack_log[1:]).mean()
    errors[n_k] = err
    print(f'{n_k} modes : erreur absolue moyenne = {err*100:.3f} vol pts (log)')

# Visualisation — comparaison surface réelle vs reconstruite (dernier jour)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

surf_actual = surf_stack_iv[-1]  # dernière surface observée

for ax_idx, n_k in enumerate([1, 2, 3]):
    proj_last = proj_matrix[-1, :n_k]
    log_recon = surf0_log.copy()
    for k in range(n_k):
        log_recon += proj_last[k] * eigenmodes[k]
    surf_recon = np.exp(log_recon)

    diff = (surf_recon - surf_actual) * 100

    im = axes[ax_idx].imshow(
        diff.T, aspect='auto', origin='lower',
        cmap='RdBu_r', vmin=-2, vmax=2,
        extent=[M_GRID[0], M_GRID[-1], TAU_GRID[0]*12, TAU_GRID[-1]*12]
    )
    plt.colorbar(im, ax=axes[ax_idx], label='Erreur (vol pts)')
    axes[ax_idx].set_xlabel('Moneyness m')
    axes[ax_idx].set_ylabel('Maturité (mois)')
    axes[ax_idx].set_title(f'Erreur reconstruction {n_k} mode(s)\n'
                            f'(RMSE = {np.sqrt(np.mean(diff**2)):.2f} vol pts)')

plt.suptitle('Section 10 — Qualité d\'ajustement : surface réelle vs reconstruite',
             fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  GÉNÉRATION DE SCÉNARIOS — DISTRIBUTION FUTURE DE LA SURFACE
# ============================================================
# Simulation sur 1 mois (21 jours) — 1000 chemins
_, surf_scenarios = simulate_factor_model(
    ou_params, eigenmodes, surf0_log,
    N_sim=21, n_paths=500, seed=99
)

# Distribution de la vol ATM à différentes maturités
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax_idx, (tau_idx, tau_label) in enumerate(
        zip([0, 2, 5], ['1M', '3M', '12M'])):

    # Vol ATM après 1 mois (horizon de simulation = 21j)
    atm_vol_T = surf_scenarios[:, -1, atm_idx, tau_idx] * 100
    vol0      = surf_stack_iv[-1, atm_idx, tau_idx] * 100

    p5, p25, p50, p75, p95 = np.percentile(atm_vol_T, [5, 25, 50, 75, 95])

    axes[ax_idx].hist(atm_vol_T, bins=50, density=True,
                      color='steelblue', alpha=0.7, edgecolor='k', lw=0.3)
    axes[ax_idx].axvline(vol0,  color='k',         lw=2.5, label=f'Actuel : {vol0:.1f}%')
    axes[ax_idx].axvline(p50,   color='firebrick',  lw=2,   ls='--', label=f'Médiane : {p50:.1f}%')
    axes[ax_idx].axvspan(p5, p95, alpha=0.15, color='orange', label='IC 90%')
    axes[ax_idx].set_xlabel(f'Vol ATM {tau_label} dans 1 mois (%)')
    axes[ax_idx].set_ylabel('Densité')
    axes[ax_idx].set_title(f'Distribution vol ATM {tau_label}\n'
                            f'IC90% = [{p5:.1f}%, {p95:.1f}%]')
    axes[ax_idx].legend(fontsize=8)

plt.suptitle('Section 10 — Scénarios futurs de la surface de vol (horizon 1 mois, 500 chemins)',
             fontweight='bold')
plt.tight_layout()
plt.show()

print('\nApplication clé (section 7.4) :')
print('  Ce framework permet de calculer des intervalles de confiance')
print('  pour la valeur future de tout portefeuille d\'options.')

---
## Section 11 — Applications : hedging Vega, risque volatilité

### 11.1 Risque Vega comme somme de contributions factorielles

Pour un portefeuille d'options $\Pi$, la sensibilité au $k$-ème facteur est :
$$\frac{\partial \Pi}{\partial x_k} = \int_A \frac{\partial \Pi}{\partial \sigma^{BS}(m,\tau)} \cdot \sigma^{BS}(m,\tau) \cdot f_k(m,\tau)\,dm\,d\tau$$

C'est le **Vega factoriel** — la somme pondérée des Vegas conventionnels.

### 11.2 Comparaison avec la règle sticky moneyness

La règle sticky moneyness correspond à $x_k(t) = \text{const}$ (pas de fluctuation) et sous-estime le risque Vega.

### 11.3 Décorrélation Vega/Delta

Les modes 2 et 3 sont peu ou pas corrélés avec le sous-jacent → le risque Vega ne peut **pas** être réduit au risque Delta (contrairement aux modèles à 1 facteur).

In [ ]:
# ============================================================
#  VEGA FACTORIEL ET RISK MANAGEMENT
# ============================================================
def compute_vega_surface(S, K_grid, tau_grid, sigma_surface, r=0., q=0.):
    """
    Calcule la surface de Vega BS pour une grille de strikes et maturités.
    sigma_surface : (N_K, N_tau) — vols implicites
    """
    vega_surf = np.zeros_like(sigma_surface)
    for i, K in enumerate(K_grid):
        for j, tau in enumerate(tau_grid):
            sigma = sigma_surface[i, j]
            if sigma > 0.001 and tau > 0.001:
                d1 = (np.log(S/K) + tau*(r-q+0.5*sigma**2)) / (sigma*np.sqrt(tau))
                vega_surf[i, j] = S * np.exp(-q*tau) * norm.pdf(d1) * np.sqrt(tau)
    return vega_surf


def factor_vega(portfolio_vegas, eigenmodes, sigma_surface):
    """
    Calcule le Vega factoriel pour chaque mode.
    portfolio_vegas : (N_M, N_T) — positions Vega du portefeuille
    Retourne : (N_modes,) — Vega factoriel
    """
    n_modes = len(eigenmodes)
    fv = np.zeros(n_modes)
    for k, f_k in enumerate(eigenmodes):
        # ∂Π/∂x_k = Σ_{m,τ} vega(m,τ) * σ(m,τ) * f_k(m,τ)
        fv[k] = np.sum(portfolio_vegas * sigma_surface * f_k)
    return fv


# Exemple de portefeuille : straddle ATM
S0 = 100.
surf_now = surf_stack_iv[-1]  # surface actuelle

# Position : long 1 call ATM + 1 put ATM pour chaque maturité
portfolio_vega = np.zeros((N_M, N_T))
for j, tau in enumerate(TAU_GRID):
    for i, m in enumerate(M_GRID):
        K = m * S0
        sigma = surf_now[i, j]
        if abs(m - 1.0) < 0.02:  # positions ATM uniquement
            portfolio_vega[i, j] = bs_vega(S0, K, tau, sigma)

# Vega factoriel
fv = factor_vega(portfolio_vega, eigenmodes, surf_now)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Surface de Vega du portefeuille
im0 = axes[0].imshow(portfolio_vega.T, aspect='auto', origin='lower',
                      cmap='viridis', extent=[M_GRID[0], M_GRID[-1],
                                               TAU_GRID[0]*12, TAU_GRID[-1]*12])
plt.colorbar(im0, ax=axes[0], label='Vega')
axes[0].set_title('Surface de Vega du portefeuille\n(straddle ATM)')
axes[0].set_xlabel('Moneyness')
axes[0].set_ylabel('Maturité (mois)')

# 2. Vega factoriel
axes[1].bar(range(1, N_MODES+1), np.abs(fv),
            color=mode_colors[:N_MODES], alpha=0.8, edgecolor='k')
axes[1].set_xlabel('Numéro du facteur')
axes[1].set_ylabel('|Vega factoriel|')
axes[1].set_title('Vega factoriel du portefeuille\n(risque volatilité par mode)')
axes[1].set_xticks([1, 2, 3])
axes[1].set_xticklabels(['Mode 1\n(Niveau)', 'Mode 2\n(Skew)', 'Mode 3\n(Butterfly)'])
for i, v in enumerate(fv):
    axes[1].text(i+1, abs(v)+0.01*max(abs(fv)), f'{v:.3f}', ha='center', fontsize=10)

# 3. Contribution de chaque facteur à la variance du P&L
nu2 = [ou_params[k]['gamma']**2 / (2*ou_params[k]['lambda'])
       for k in range(N_MODES)]
pnl_var_contrib = [fv[k]**2 * nu2[k] for k in range(N_MODES)]
total_pnl_var   = sum(pnl_var_contrib)

axes[2].pie([v/total_pnl_var*100 for v in pnl_var_contrib],
            labels=['Niveau\n({:.1f}%)'.format(pnl_var_contrib[0]/total_pnl_var*100),
                    'Skew\n({:.1f}%)'.format(pnl_var_contrib[1]/total_pnl_var*100),
                    'Butterfly\n({:.1f}%)'.format(pnl_var_contrib[2]/total_pnl_var*100)],
            colors=mode_colors[:N_MODES], startangle=90,
            autopct='%1.1f%%', pctdistance=0.75)
axes[2].set_title('Décomposition de la variance du P&L Vega\n(contribution de chaque facteur)')

plt.suptitle('Section 11 — Hedging Vega : décomposition factorielle du risque',
             fontweight='bold')
plt.tight_layout()
plt.show()

print('\nConclusion (section 7.1) :')
print('  Le risque Vega a plusieurs sources indépendantes → il ne se réduit pas au Delta')
print(f'  Variance P&L : Mode 1 = {pnl_var_contrib[0]/total_pnl_var*100:.1f}%,',
      f'Mode 2 = {pnl_var_contrib[1]/total_pnl_var*100:.1f}%,',
      f'Mode 3 = {pnl_var_contrib[2]/total_pnl_var*100:.1f}%')

---
## Section 12 — Dashboard complet & Synthèse

In [ ]:
# ============================================================
#  DASHBOARD FINAL — 9 GRAPHIQUES
# ============================================================
fig = plt.figure(figsize=(20, 15))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.42, wspace=0.38)

# 1. Surface moyenne
ax1 = fig.add_subplot(gs[0, 0], projection='3d')
ax1.plot_surface(M_MESH, TAU_MESH*12, mean_iv*100, cmap='viridis', alpha=0.85)
ax1.set_xlabel('m', fontsize=7)
ax1.set_ylabel('τ(M)', fontsize=7)
ax1.set_zlabel('%', fontsize=7)
ax1.set_title('Surface moyenne\nSX5E', fontsize=9)
ax1.view_init(25, -55)

# 2. Écart-type quotidien
ax2 = fig.add_subplot(gs[0, 1])
im2 = ax2.imshow(std_logvar.T*100, aspect='auto', origin='lower', cmap='hot',
                  extent=[M_GRID[0], M_GRID[-1], TAU_GRID[0]*12, TAU_GRID[-1]*12])
plt.colorbar(im2, ax=ax2, shrink=0.8)
ax2.set_title('Std quotidienne (vol pts)\n→ fixed smile rejeté', fontsize=9)
ax2.set_xlabel('m')
ax2.set_ylabel('τ (M)')

# 3. Variance expliquée
ax3 = fig.add_subplot(gs[0, 2])
ax3.bar(range(1, n_show+1), var_expl[:n_show], color='steelblue', alpha=0.8, edgecolor='k', lw=0.5)
ax3.set_xlabel('Rang du mode')
ax3.set_ylabel('Variance (%)')
ax3.set_title('Valeurs propres\n(décroissance rapide)', fontsize=9)
ax3.set_xlim(0.5, 8.5)

# 4. Mode 1 (niveau)
ax4 = fig.add_subplot(gs[1, 0])
for j_tau in [0, 2, 5]:
    ax4.plot(M_GRID, eigenmodes[0][:, j_tau], lw=2, label=f'{TAU_GRID[j_tau]*12:.0f}M')
ax4.axhline(0, color='k', lw=0.7)
ax4.set_title(f'Mode 1 — NIVEAU\n({var_expl[0]:.1f}% variance)', fontsize=9)
ax4.set_xlabel('m')
ax4.legend(fontsize=7)

# 5. Mode 2 (skew)
ax5 = fig.add_subplot(gs[1, 1])
for j_tau in [0, 2, 5]:
    ax5.plot(M_GRID, eigenmodes[1][:, j_tau], lw=2, label=f'{TAU_GRID[j_tau]*12:.0f}M')
ax5.axhline(0, color='k', lw=0.7)
ax5.axvline(1., color='gray', lw=0.7, ls='--')
ax5.set_title(f'Mode 2 — SKEW\n({var_expl[1]:.2f}% variance)', fontsize=9)
ax5.set_xlabel('m')
ax5.legend(fontsize=7)

# 6. Mode 3 (butterfly)
ax6 = fig.add_subplot(gs[1, 2])
for j_tau in [0, 2, 5]:
    ax6.plot(M_GRID, eigenmodes[2][:, j_tau], lw=2, label=f'{TAU_GRID[j_tau]*12:.0f}M')
ax6.axhline(0, color='k', lw=0.7)
ax6.set_title(f'Mode 3 — BUTTERFLY\n({var_expl[2]:.2f}% variance)', fontsize=9)
ax6.set_xlabel('m')
ax6.legend(fontsize=7)

# 7. Projection Mode 1 dans le temps
ax7 = fig.add_subplot(gs[2, 0])
ax7.plot(proj_df.index, proj_df['x1'], 'steelblue', lw=1.2, alpha=0.85)
ax7.set_title('Proj. Mode 1 $x_1(t)$\n(AR(1)/OU, mean-reverting)', fontsize=9)
ax7.xaxis.set_tick_params(rotation=30, labelsize=7)
ax7.axhline(0, color='k', lw=0.7)

# 8. Corrélation Mode 1 / sous-jacent
ax8 = fig.add_subplot(gs[2, 1])
dx1 = proj_df['x1'].diff().dropna()
common = dx1.index.intersection(spot_ret_daily.index)
ax8.scatter(spot_ret_daily.loc[common]*100, dx1.loc[common],
            alpha=0.2, s=5, color='steelblue')
corr1 = np.corrcoef(dx1.loc[common], spot_ret_daily.loc[common])[0,1]
ax8.set_title(f'Mode 1 vs rendement S\ncorr = {corr1:.3f} (Levier)', fontsize=9)
ax8.set_xlabel('Rendement sous-jacent (%)')
ax8.set_ylabel('$\\Delta x_1$')

# 9. Vega factoriel
ax9 = fig.add_subplot(gs[2, 2])
ax9.bar(range(1, N_MODES+1), np.abs(fv),
        color=mode_colors[:N_MODES], alpha=0.8, edgecolor='k')
ax9.set_title('Vega factoriel\n(straddle ATM)', fontsize=9)
ax9.set_xticks([1,2,3])
ax9.set_xticklabels(['Niv.','Skew','Bfly'], fontsize=8)

plt.suptitle('Dashboard Cont & da Fonseca (2002) — Dynamics of Implied Vol Surfaces\nSX5E',
             fontsize=14, fontweight='bold')
plt.savefig('cont_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard sauvegardé.')

In [ ]:
# ============================================================
#  BILAN QUANTITATIF FINAL
# ============================================================
print('=' * 72)
print('  BILAN — Dynamics of Implied Volatility Surfaces')
print('  Cont & da Fonseca (2002) — Application SX5E')
print('=' * 72)

print(f'''
  DONNÉES
  ──────────────────────────────────────────────────────────────
  Indice         : SX5E (EuroStoxx 50)
  Période        : {dates_sorted[0]} → {dates_sorted[-1]}
  N dates        : {N_dates}
  Grille (m, τ)  : {N_M}×{N_T} = {N_pts} points
  Moneyness      : {M_GRID[0]:.0%} → {M_GRID[-1]:.0%}
  Maturités      : 1M → 12M

  DÉCOMPOSITION DE KARHUNEN-LOÈVE
  ──────────────────────────────────────────────────────────────
  Mode 1 (Niveau)     : {var_expl[0]:.2f}%  (papier SP500 : 94%, FTSE : 96%)
  Mode 2 (Skew)       : {var_expl[1]:.2f}%  (papier SP500 : 3%)
  Mode 3 (Butterfly)  : {var_expl[2]:.2f}%  (papier SP500 : 0.8%)
  ─────────────────────────────────────────────────────────────
  3 modes cumulés     : {var_cum[2]:.2f}%  (papier : >95%)
''')

print('  STATISTIQUES DES COMPOSANTES PRINCIPALES')
print('  ──────────────────────────────────────────────────────────────')
for k in range(N_MODES):
    x_k   = proj_df[f'x{k+1}'].dropna()
    dx_k  = x_k.diff().dropna()
    lam   = ou_params[k]['lambda']
    mr    = 1./lam if lam > 0 else np.nan
    common = dx_k.index.intersection(spot_ret_daily.index)
    corr  = np.corrcoef(dx_k.loc[common], spot_ret_daily.loc[common])[0,1] if len(common) > 10 else np.nan

    print(f'  Mode {k+1} [{["Niveau","Skew","Butterfly"][k]}] :')
    print(f'    Std journalière      = {x_k.std():.4f}')
    print(f'    Kurtosis (Fisher)    = {scipy_kurtosis(x_k):.2f}')
    print(f'    Mean-reversion       ≈ {mr:.1f} jours')
    print(f'    Corr sous-jacent     = {corr:.3f}')
    print(f'    λ (OU)               = {lam:.4f}/j')

print(f'''
  CONCLUSIONS PRINCIPALES
  ──────────────────────────────────────────────────────────────
  1. Un modèle à 2-3 facteurs explique >95% des fluctuations quotidiennes
  2. Les 3 modes sont interprétables : Niveau, Skew, Butterfly
  3. Les composantes suivent des processus AR(1)/OU mean-reverting
     avec des temps de retour ≈ 1 mois (≈ lifetime des options courtes)
  4. Le mode 1 (niveau) est fortement corrélé négativement avec le S
     → effet levier
  5. Les modes 2 & 3 sont peu corrélés avec le S
     → le risque Vega NE SE RÉDUIT PAS au risque Delta
  6. Le modèle sticky moneyness (fixed smile) est clairement rejeté
     (std quotidienne pouvant atteindre {std_logvar.max()*100:.1f}%)
  7. Std max = {std_logvar.max()*100:.1f}% ≈ {std_logvar.max()/mean_iv.mean()*100:.0f}% de la vol moyenne
     → impact matériel sur les portefeuilles d\'options
''')

print('=' * 72)

---

## Résumé méthodologique

```
Données brutes
    │
    ▼
Filtrage (m ∈ [0.5, 1.5], τ ∈ [2w, 1.5Y])
    │
    ▼
Lissage Nadaraya-Watson → surfaces journalières lisses Ît(m,τ)
    │
    ▼
Variations log-quotidiennes : Xt(m,τ) = ln Ît(m,τ) - ln Ît-1(m,τ)
    │
    ▼
Matrice de covariance K(x1,x2) = cov(U(x1), U(x2))
    │
    ▼
Décomposition KL (Galerkin/SVD) → modes propres fk + variances νk²
    │
    ▼
Projections xk(t) = <Xt, fk>  →  processus AR(1)/OU
    │
    ▼
Modèle factoriel : ln It(m,τ) = X0(m,τ) + Σk xk(t) fk(m,τ)
    │
    ▼
Applications : hedging Vega, scénarios, risk management
```

---

## Références

- **Cont, R. & da Fonseca, J. (2002)** — *Dynamics of Implied Volatility Surfaces*, Quantitative Finance, 2, 45–60
- **Skiadopoulos, G., Hodges, S. & Clelow, L. (2000)** — *Dynamics of the S&P 500 implied volatility surface*, Review of Derivatives Research, 3, 263–82
- **Fengler, M., Härdle, W. & Villa, C. (2000)** — *Common principal components approach for PCA of IV smiles*
- **Balland, P. (2002)** — *Deterministic implied volatility surfaces*, Quantitative Finance, 2, 31–44
- **Avellaneda, M. & Zhu, Y. (1997)** — *An E-ARCH model for the term structure of implied volatility of FX options*
- **Derman, E. (1998)** — *Regimes of volatility*, Risk

---
*Notebook réalisé pour l'implémentation complète de Cont & da Fonseca (2002) — Anthropic Claude*